In [39]:
import altair as alt    
import polars as pl 

In [100]:
df = (
    pl.scan_parquet(f's3://aind-scratch-data/ben.hardcastle/num_threads_benchmark/')
    .filter(
        pl.col('library benchmarked') == 'openblas',
    )
    .collect()
)

---
### multithreading in `BLAS` affects performance of basic matrix operations in `numpy`
- number of threads can be set via `OPENBLAS_NUM_THREADS`
- benchmark uses `np.dot()` on two square matrices

In [118]:
(
    df
    .filter(
        # pl.col('is_pipeline') == True,
        # pl.col('matrix size') == 1000,
    )
    .plot.line(
        x='OPENBLAS_NUM_THREADS:O',
        y=alt.Y('wall time:Q', title='wall time (s)'),
        color='co cpu count:O',
        column='matrix size:O',
        detail='os cpu count:O',
        row='is_pipeline',
    )
    .resolve_scale(
        x='independent',
        y='independent',
    )
    .properties(
        title=alt.TitleParams(
            text='multithreading in BLAS affects numpy performance',
            subtitle=[
                "- benchmark uses `np.dot()` on two square matrices",
                "- number of threads are set via environment variable",
                "- upper lim is number of CPUs reported by the OS with nproc",   
            ],
            anchor='start',
            orient='bottom',
            offset=20,
        ),
        height=200,
        width=200,
    )
)

alt.Chart(...)

- we can't easily tell what the default number of threads used by BLAS (can be set when compiled, or
  set automatically)
- default performance appears to be worse than setting manually 
- cannot rely on reading number of CPUs reported by OS

In [102]:
(
    df.plot.line(
        x=alt.X('co cpu count:O').title('cores requested'),
        y=alt.Y('os cpu count:O').title('os.cpu_count()'),
        facet='is_pipeline',
    )
    .properties(
        title=alt.TitleParams(
            text='CPU count reported by Ubuntu does not change with size of machine requested in CO',
            anchor='start',
            orient='bottom',
            offset=20,
            fontSize=12,
        ),
    )
)

alt.Chart(...)

- setting Nthreads >> Ncores is very bad
- setting Nthreads << Ncores is quite bad
- not setting can be very bad if cores available << cores on machine

In [103]:
def norm_time()  -> pl.Expr:
    over_cols = ['is_pipeline', 'library benchmarked', 'co cpu count', 'os cpu count', 'matrix size']
    # normalize wall time to value when nthreads == num cpu cores available
    return (
        pl.when(pl.col('library benchmarked') == 'openblas')
        .then(
            (pl.col('wall time') / pl.col('wall time').filter(pl.col('OPENBLAS_NUM_THREADS') == pl.col('co cpu count')).first()).over(over_cols)
        )
        .when(pl.col('library benchmarked') == 'mkl')
        .then(
            (pl.col('wall time') / pl.col('wall time').filter(pl.col('MKL_NUM_THREADS') == pl.col('co cpu count')).first()).over(over_cols)
        )
        .otherwise(pl.lit(None))
    )

wall_time_chart = (
    df
    .with_columns(
        norm_time().alias('normalized wall time'),
    )
    .filter(
        pl.col('matrix size') >= 1000,
    )
    .plot.rect(
        x='OPENBLAS_NUM_THREADS:O',
        y=alt.Y('co cpu count:O').scale(reverse=True),
        color=alt.Color('wall time:Q'),
        column='matrix size:O',
        detail='os cpu count:O',
        row='is_pipeline',
    )

)
(
    wall_time_chart
    .resolve_scale(
        x='independent',
        color='independent',
    )
)

alt.Chart(...)

- setting N-threads = N-cores requested in CO generally does not make performance worse
    - NOTE: for 32/64 cores requested in pipeline, `CO_CPUS` method does not work

- the exception is pipelines (not sure whether 1 requested cpu == 1 logical or 1 physical core)

In [109]:
(
    wall_time_chart
    .encode(
        color=alt.Color('normalized wall time:Q').scale(type='log', scheme="redblue", domainMin=0.1, domainMid=1, domainMax=10, reverse=True),
    )
    .resolve_scale(
        x='independent',
    )
)

alt.Chart(...)

- setting N-threads == N-cores requested 

In [116]:
(
    df
    .with_columns(
        norm_time().alias('normalized wall time'),
    )
    .filter(
        (pl.col('normalized wall time') == 1) | (pl.col('OPENBLAS_NUM_THREADS').is_null())
    )
    .with_columns(
        pl.when(pl.col('OPENBLAS_NUM_THREADS').is_null())
        .then(pl.lit('not set'))
        .when(pl.col('normalized wall time') == 1)
        .then(pl.lit('CO_CPUS'))
        .otherwise(pl.col('OPENBLAS_NUM_THREADS').cast(pl.Utf8))
        .alias('OPENBLAS_NUM_THREADS')
    )
    .plot.line(
        x=alt.X('OPENBLAS_NUM_THREADS:N').scale(reverse=True),
        y=alt.Y('normalized wall time:Q').scale(type='log'),
        color='co cpu count:O',
        row='is_pipeline',
        column='matrix size:O',
    )
    .properties(
        width=100,
        height=150,
    )
)

alt.Chart(...)

## questions
- does OPENMP setting work (recommended by numpy, possibly more general across OSs)?
- why OPENBLAS_NUM_THREADS = 8 on a 1 core machine in AWS Batch 5x faster than OPENBLAS_NUM_THREADS = 1